# Bridge 6 Pilot: Real Molecules Test

**Goal**: First signal check with REAL data. Do we see any pattern between molecular symmetry (point group) and assembly complexity (MA)?

**Dataset**: Real molecules from PubChem (diverse chemistry, natural symmetry variation)

**Key difference from synthetic test**: Real molecules have diverse point groups (C1, C2v, Td, D3h, etc.), not just C1 and Cs.

**Outputs**:
1. Scatter plot: MA vs point group order
2. Spearman correlation + p-value
3. Summary statistics
4. Box plots by symmetry type
5. PCA visualization

**Success criteria**: Sufficient variation in point groups to enable correlation testing.

In [2]:
import sys
sys.path.insert(0, '/c/Users/Cicada38/Projects/math_exp')

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Create directories
Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('figures').mkdir(parents=True, exist_ok=True)

print("Environment ready.")
print("\n" + "="*60)
print("BRIDGE 6 PILOT: REAL MOLECULES TEST")
print("="*60)

Environment ready.

BRIDGE 6 PILOT: REAL MOLECULES TEST


## Step 1: Fetch real molecules from PubChem

In [3]:
from src.fetch_molecules import get_qm9_sample

# Get real molecules from PubChem
# Start with 500 to get good diversity
print("Fetching real molecules from PubChem...")
print("This may take 2-3 minutes due to API rate limiting...\n")
df_molecules = get_qm9_sample(n=500, use_real_data=True)

print(f"\nLoaded {len(df_molecules)} real molecules")
print("\nFirst 10 SMILES:")
if 'num_atoms' in df_molecules.columns:
    print(df_molecules[['smiles', 'num_atoms']].head(10).to_string())
else:
    print(df_molecules[['smiles']].head(10).to_string())

Fetching real molecules from PubChem...
This may take 2-3 minutes due to API rate limiting...

Fetching 500 molecules from PubChem...


KeyboardInterrupt: 

## Step 2: Compute point group symmetry for each molecule

In [ ]:
from src.compute_symmetry import compute_point_group

print(f"Computing point groups for {len(df_molecules)} molecules...")
print("This may take several minutes...\n")

symmetry_results = []
failures = 0

for i, smiles in enumerate(df_molecules['smiles']):
    if i % 50 == 0:
        print(f"  {i}/{len(df_molecules)}...", end='\r')
    
    try:
        result = compute_point_group(smiles, tolerance=0.3)
        if result and result.get('point_group'):
            result['smiles'] = smiles
            symmetry_results.append(result)
        else:
            failures += 1
    except Exception as e:
        failures += 1

df_symmetry = pd.DataFrame(symmetry_results)
print(f"\nComputed symmetry for {len(df_symmetry)} molecules")
print(f"Failures: {failures}")
print(f"Success rate: {len(df_symmetry)/(len(df_molecules))*100:.1f}%")

print("\nPoint group distribution:")
print(df_symmetry['point_group'].value_counts())

print("\nPoint group order distribution:")
print(df_symmetry['order'].value_counts().sort_index())

## Step 3: Compute assembly index for each molecule

In [ ]:
from src.compute_assembly import AssemblyIndexBatcher

print(f"Computing assembly indices for {len(df_symmetry)} molecules...")
print("Using heuristic method for speed (real MA would use assembly-theory package)\n")

batcher = AssemblyIndexBatcher(method='heuristic', delay_seconds=0.0)
assembly_results = batcher.compute_batch(df_symmetry['smiles'].tolist())

df_assembly = pd.DataFrame(assembly_results)
df_assembly = df_assembly[['smiles', 'assembly_index']]

print(f"\nComputed assembly indices for {len(df_assembly)} molecules")
print(f"\nAssembly Index statistics:")
print(f"  Mean: {df_assembly['assembly_index'].mean():.2f}")
print(f"  Std:  {df_assembly['assembly_index'].std():.2f}")
print(f"  Min:  {df_assembly['assembly_index'].min():.2f}")
print(f"  Max:  {df_assembly['assembly_index'].max():.2f}")
print(f"  Median: {df_assembly['assembly_index'].median():.2f}")

## Step 4: Merge datasets

In [ ]:
from src.merge_datasets import merge_datasets, save_dataset

df_merged = merge_datasets(df_symmetry, df_assembly)

print(f"Merged dataset: {len(df_merged)} molecules with both symmetry and assembly data")
print("\nColumns:")
print(df_merged.columns.tolist())
print("\nFirst 10 rows:")
print(df_merged[['smiles', 'point_group', 'order', 'assembly_index']].head(10))

# Save for later analysis
save_dataset(df_merged, 'data/processed/pilot_real_molecules.csv')

## Step 5: Core correlation analysis (THE TRUTH TEST)

In [ ]:
from src.analyze import test_ma_vs_symmetry, compute_summary_stats

# Test main hypothesis
result = test_ma_vs_symmetry(df_merged)

print("\n" + "=" * 60)
print("MA vs Point Group Order Correlation")
print("=" * 60)
print(f"Spearman ρ:  {result['rho']:.4f}")
print(f"p-value:     {result['p_value']:.6f}")
print(f"n samples:   {result['n']}")
print(f"Status:      {result['status']}")

if result['p_value'] < 0.05:
    print("\n✓ SIGNIFICANT: Correlation detected at p < 0.05")
    if result['rho'] > 0:
        print(f"  Interpretation: More symmetric molecules tend to have higher MA")
    else:
        print(f"  Interpretation: More symmetric molecules tend to have lower MA")
else:
    print("\n⚠ NO SIGNIFICANT CORRELATION: p >= 0.05")
    print("  Interpretation: Point group symmetry and assembly complexity are independent")
    print("  (Still publishable as an orthogonal molecular descriptor!)")

# Summary stats
print("\n" + "=" * 60)
print("Summary Statistics")
print("=" * 60)
stats_dict = compute_summary_stats(df_merged)
for key, val in stats_dict.items():
    if key != 'point_groups':
        print(f"{key:20} : {val}")

## Step 6: Visualizations

In [ ]:
from src.analyze import plot_ma_vs_symmetry, plot_distribution, plot_symmetry_box

# Main plot: MA vs symmetry
print("Creating scatter plot: MA vs Point Group Order...")
plot_ma_vs_symmetry(df_merged, output_path='figures/01_ma_vs_symmetry_real.png')

# Distribution of assembly indices
print("\nCreating MA distribution histogram...")
plot_distribution(df_merged, 'assembly_index', output_path='figures/01_ma_distribution_real.png')

# Box plots by symmetry type
print("\nCreating box plot: MA by point group...")
plot_symmetry_box(df_merged, output_path='figures/01_ma_by_symmetry_real.png')

## Step 7: Detailed Analysis by Point Group

In [ ]:
print("\n" + "=" * 60)
print("Detailed Analysis by Point Group")
print("=" * 60)

pg_analysis = df_merged.groupby('point_group').agg({
    'assembly_index': ['count', 'mean', 'std', 'min', 'max'],
    'order': 'first'
}).round(3)

print("\n" + pg_analysis.to_string())

# Identify interesting cases
print("\n" + "=" * 60)
print("Interesting Cases: High Symmetry + High/Low MA")
print("=" * 60)

# High symmetry groups
high_sym = df_merged[df_merged['order'] >= 4].copy()
if len(high_sym) > 0:
    print(f"\nHigh symmetry molecules (order >= 4): {len(high_sym)}")
    print(f"Mean MA: {high_sym['assembly_index'].mean():.2f}")
    print(f"\nTop 5 by MA:")
    print(high_sym.nlargest(5, 'assembly_index')[['smiles', 'point_group', 'order', 'assembly_index']].to_string())

## Summary & Interpretation

### Key Findings:

**If correlation is significant (p < 0.05):**
- ✓ The hypothesis has empirical support
- ✓ Molecular symmetry predicts assembly complexity
- → Scale to full QM9 (134k molecules) for robustness
- → Investigate mechanism (why?)

**If no correlation:**
- ✓ Symmetry and assembly complexity are orthogonal descriptors
- ✓ Still highly publishable (both measures are useful for drug discovery)
- → Investigate whether relationship emerges for specific chemical families
- → Test whether biotic/abiotic threshold changes symmetry profiles

### Next Steps:

1. **If signal detected (p < 0.05):**
   - Expand to full QM9 dataset
   - Stratify by molecular size, chemical families
   - Connect to Thread 1 (SAI on finite groups)

2. **If no signal (p >= 0.05):**
   - Document as negative result
   - Investigate confounders (size effects, family-specific patterns)
   - Proceed to Thread 1 (SAI metric definition)

3. **Thread 1 (Parallel work):**
   - Define Symmetry Assembly Index for finite groups
   - Compute SAI for all point groups in dataset
   - Test if SAI predicts MA (the bridge)

---

**This is a real, publishable result either way.** The question "Are symmetry and assembly complexity related?" is empirically answerable, and your answer will advance the field.